In [ ]:

# =========================== REPO PATHS ===========================
from pathlib import Path
def _find_project_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        if (candidate / "code" / "data").exists():
            return candidate
    if cwd.name == "code" and (cwd / "data").exists():
        return cwd.parent
    return cwd

PROJECT_ROOT = _find_project_root()
DATA_DIR = PROJECT_ROOT / "code" / "data"

def _resolve_csv_path(filename: str) -> str:
    path = Path(filename)
    if path.exists():
        return str(path)
    for candidate in (
        DATA_DIR / path.name,
        PROJECT_ROOT / path.name,
        PROJECT_ROOT / "data" / path.name,
    ):
        if candidate.exists():
            return str(candidate)
    return str(DATA_DIR / path.name)

def _project_out(*parts: str) -> Path:
    return PROJECT_ROOT.joinpath(*parts)

def _pick_existing_path(*candidates: str) -> Path:
    paths = []
    for candidate in candidates:
        path = Path(candidate)
        paths.append(path if path.is_absolute() else PROJECT_ROOT / path)
    for path in paths:
        if path.exists():
            return path
    return paths[0]


from pathlib import Path
from typing import Dict, List, Tuple
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import re
from pathlib import Path
from typing import List

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
try:
    from adjustText import adjust_text
except ImportError:
    def adjust_text(*args, **kwargs):
        return []

CSV_FILE = _pick_existing_path(
    "tables_HEADDIST_no_index/headdist_bootstrap/headdist_raw_iso_gpt2.csv",
    "gpt2_no_index/tables_HEADDIST_no_index/headdist_bootstrap/headdist_raw_iso_gpt2.csv",
)
OUT_DIR = _project_out("plot_POS")
OUT_DIR.mkdir(parents=True, exist_ok=True)

LABELS = {
    "iso": "Isotropy", "spect": "Spectral Ratio", "rand": "RandCos |μ|",
    "sf": "Spectral Flatness", "vmf_kappa": "vMF κ",
    "erank": "Effective Rank", "pr": "Participation Ratio", "stable_rank": "Stable Rank",
    "lpca95": "lPCA95", "lpca99": "Linear ID", "lpca": "lPCA FO", "pca99": "Linear ID",
    "twonn": "TwoNN ID", "gride": "Nonlinear ID", "mom": "MOM", "tle": "TLE",
    "lpca99_skdim": "Linear ID",
    "corrint": "CorrInt", "fishers": "FisherS", "mle": "MLE", "mada": "MADA", "knn": "KNN",
}

# ----------------- Helpers -----------------
def make_class_palette(classes: List[str], cmap_name: str = "viridis"):
    """
    Map each class to a color.
    - Continuous palettes (viridis, plasma, rocket, etc.) are sampled smoothly.
    - Discrete palettes (tab20, tab10, Set2, Paired, etc.) use their fixed colors.
    """
    ordered = sorted(classes)
    n = len(ordered)

    # --- Handle discrete palettes explicitly ---
    DISCRETE = {"tab20", "tab10", "tab20b", "tab20c", "Set1", "Set2", "Set3",
                "Paired", "Accent", "Dark2", "Pastel1", "Pastel2"}

    if cmap_name in DISCRETE:
        base = sns.color_palette(cmap_name)  # fixed categorical colors
        m = len(base)
        return {cls: base[i % m] for i, cls in enumerate(ordered)}

    # --- Otherwise: continuous colormap ---
    cmap = plt.get_cmap(cmap_name)
    return {
        cls: cmap(i / max(n - 1, 1))
        for i, cls in enumerate(ordered)
    }


def _sanitize(s: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", str(s)).strip("_")


def choose_label_position_for_class(
    cls: str,
    classes: List[str],
    xy_dict: dict,
    xmin: float,
    xmax: float,
    n_candidates: int = 25,
):
    """
    For one class, scan along its line and pick a point where:
    - it is far from other lines (large vertical separation), and
    - it's not too close to the edges in x.
    """
    x, y = xy_dict[cls]
    n = len(x)
    if n == 0:
        return None, None

    # Candidate indices along the line
    if n <= n_candidates:
        cand_idx = np.arange(n)
    else:
        step = max(1, n // n_candidates)
        # avoid the very first/last points
        cand_idx = np.arange(step, n - step, step)
        if len(cand_idx) == 0:
            cand_idx = np.arange(n)

    scores = []

    for idx in cand_idx:
        x_c = x[idx]
        y_c = y[idx]

        # 1) Distance to other lines at this x (approx via nearest neighbor in x)
        min_dist = np.inf
        for other in classes:
            if other == cls:
                continue
            x2, y2 = xy_dict[other]
            if len(x2) == 0:
                continue
            j = np.argmin(np.abs(x2 - x_c))
            d = abs(y_c - y2[j])
            if d < min_dist:
                min_dist = d
        if not np.isfinite(min_dist):
            min_dist = 0.0

        # 2) Edge penalty: prefer middle of x-range over the edges
        if np.isfinite(xmin) and np.isfinite(xmax) and xmax > xmin:
            u = (x_c - xmin) / (xmax - xmin)  # in [0,1]
            # parabola: 1 in the center (0.5), 0 at edges
            edge_score = 1.0 - 4.0 * (u - 0.5) ** 2
            if edge_score < 0:
                edge_score = 0.0
        else:
            edge_score = 1.0

        # Combine: we want large separation and prefer the center.
        score = min_dist * (0.5 + 0.5 * edge_score)
        scores.append(score)

    scores = np.asarray(scores)
    if scores.size == 0:
        # fallback: middle of the line
        mid = n // 2
        return x[mid], y[mid]

    best_idx = cand_idx[int(np.argmax(scores))]
    return x[best_idx], y[best_idx]


# ----------------- Load data -----------------
df = pd.read_csv(CSV_FILE)

# Basic checks
required_cols = {"class", "layer", "mean"}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f"CSV missing required columns: {missing}")

# Types
df["layer"] = pd.to_numeric(df["layer"], errors="coerce")
if "ci_low" in df.columns:
    df["ci_low"] = pd.to_numeric(df["ci_low"], errors="coerce")
if "ci_high" in df.columns:
    df["ci_high"] = pd.to_numeric(df["ci_high"], errors="coerce")

# Meta if present (else fallbacks)
metric = (
    df["metric"].dropna().unique()[0]
    if "metric" in df.columns and df["metric"].notna().any()
    else "metric"
)
model = (
    df["model"].dropna().unique()[0]
    if "model" in df.columns and df["model"].notna().any()
    else "model"
)
subset = (
    df["subset"].dropna().unique()[0]
    if "subset" in df.columns and df["subset"].notna().any()
    else "subset"
)

ylabel = LABELS.get(str(metric), str(metric).upper())
title = "GPT"

classes = sorted(df["class"].dropna().unique())
print(f"#classes: {len(classes)}")

xy_dict = {}
ci_dict = {}

for cls in classes:
    sub = df[df["class"] == cls].sort_values("layer")
    x = sub["layer"].to_numpy(dtype=float)
    y = sub["mean"].to_numpy(dtype=float)

    mask_xy = np.isfinite(x) & np.isfinite(y)
    x = x[mask_xy]
    y = y[mask_xy]

    xy_dict[cls] = (x, y)

    if {"ci_low", "ci_high"}.issubset(sub.columns):
        lo = sub["ci_low"].to_numpy(dtype=float)[mask_xy]
        hi = sub["ci_high"].to_numpy(dtype=float)[mask_xy]
        ci_dict[cls] = (lo, hi)

# Global x-range for edge penalty
all_x = np.concatenate(
    [xy_dict[c][0] for c in classes if xy_dict[c][0].size > 0]
) if classes else np.array([])
if all_x.size > 0:
    xmin_global = float(np.nanmin(all_x))
    xmax_global = float(np.nanmax(all_x))
else:
    xmin_global, xmax_global = 0.0, 1.0

palette = make_class_palette(classes, cmap_name="coolwarm")  # or "tab20", etc.

# ----------------- Plot -----------------
sns.set_style("darkgrid")
sns.set_context("paper", font_scale=2.5)

plt.figure(figsize=(9, 5))
ax = plt.gca()
ax.margins(x=0.02)

#ax.set_xlabel("Layer")
#ax.set_ylabel(ylabel)
ax.set_title(title)

texts = []  # label artists for adjust_text

for cls in classes:
    x, y = xy_dict[cls]
    if x.size == 0:
        continue

    color = palette.get(cls, None)
    pretty_label = LABELS.get(str(cls), str(cls))

    # Main line
    ax.plot(x, y, lw=1.8, label=pretty_label, color=color)

    # CI shading, if available
    if cls in ci_dict:
        lo, hi = ci_dict[cls]
        mask_ci = np.isfinite(lo) & np.isfinite(hi)
        if mask_ci.any():
            ax.fill_between(
                x[mask_ci], lo[mask_ci], hi[mask_ci],
                alpha=0.15, color=color
            )

    # ---- Choose best label position for this line ----
    x_lab, y_lab = choose_label_position_for_class(
        cls, classes, xy_dict, xmin_global, xmax_global
    )
    if x_lab is None:
        continue

    t = ax.text(
        x_lab,
        y_lab,
        pretty_label,
        color=color,
        fontsize=15,
        va="center",
        ha="center",
        bbox=dict(
            boxstyle="round,pad=0.2",
            facecolor="white",
            edgecolor="none",
            alpha=0.7,
        ),
    )
    texts.append(t)

# --------------- De-overlap the labels vertically ----------------
if texts:
    # move labels only in y-direction so each one stays over "its" x-position
    adjust_text(texts, only_move={'text': 'y'}, ax=ax)

# Legend optional now; labels are inline
# ncol = 3 if len(classes) > 12 else 2
# ax.legend(ncol=ncol, fontsize="small", frameon=True)
plt.tight_layout()

out_name = f"fromcsv_{_sanitize(subset)}_{_sanitize(metric)}_{_sanitize(model)}.pdf"
out_path = OUT_DIR / out_name
plt.savefig(out_path, dpi=300)
plt.close()

print(f"[ok] Saved plot -> {out_path}")


In [ ]:
fig_legend = plt.figure(figsize=(12, 1.2))  # wide and short; adjust as needed

# Create dummy figure and plot nothing, just extract legend handles
dummy_fig, dummy_ax = plt.subplots()
handles, labels = [], []
for cls in classes:
    handles.append(plt.Line2D([0], [0], color=palette[cls], lw=20))
    labels.append(cls)

legend = fig_legend.legend(
    handles, labels,
    ncol=6,          
    frameon=True,             
    fontsize=60,
    title_fontsize=15,
    loc="center",
)

legend.get_frame().set_edgecolor("black")
legend.get_frame().set_linewidth(0.8)

# remove all axes
fig_legend.gca().axis("off")

legend_out = OUT_DIR / f"legend_{_sanitize(subset)}_{_sanitize(metric)}.pdf"
fig_legend.savefig(legend_out, dpi=300, bbox_inches="tight")
plt.close(fig_legend)

print(f"[ok] Saved legend -> {legend_out}")


### All tokens

In [ ]:
import pandas as pd

df = pd.read_csv(_pick_existing_path(
    "metrics_all_token/tables/alltokens_lpca99.csv",
    "metrics_all_tokens/tables/alltokens_lpca99.csv",
    "plots_extra/metrics/all_tokens/tables/alltokens_lpca99.csv",
))
print(df.head())


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.font_manager import FontProperties

name_map = {
    "bert-base-uncased": "bert",
    "openai-community/gpt2": "gpt2"
}

sns.set_style("darkgrid")
sns.set_context("paper", font_scale=2.5)

plt.figure(figsize=(9, 5))
ax = plt.gca()
ax.margins(x=0)

for (model, rep), sub in df.groupby(["model", "word_rep_mode"]):
    sub = sub.sort_values("layer")
    nice_model = name_map.get(model, model)
    label = f"{nice_model} / {rep}"

    ax.plot(sub["layer"], sub["mean"], marker="o", label=label)
    ax.fill_between(sub["layer"], sub["ci_low"], sub["ci_high"], alpha=0.2)

ax.set_xlabel("Layer")
ax.set_ylabel("Linear ID")

legend_fp = FontProperties(family="DejaVu Sans Mono", size=11)
leg = ax.legend(
    loc="upper right",
    prop=legend_fp,
    frameon=True,
    fancybox=True,
    framealpha=0.9,
    borderpad=0.25,
    labelspacing=0.2,
    handlelength=1.2,
    handletextpad=0.4,
    markerscale=0.85,
)
leg.get_frame().set_linewidth(0.6)

plt.tight_layout()
out_name = "all_LID.pdf"
plt.savefig(out_name, dpi=300)
plt.show()
